# EDA conjunto — Data4Cyber sin S0 ni S3- 

Análisis exploratorio sobre las ** tablas de proceso** del release, unidas en un solo
`DataFrame` con la columna `escenario`.

**Cómo se ejecuta.** La celda 0 resuelve sola la ruta de datos: monta Drive si corre en
Colab y cae a `datasets/` si corre local. No hay rutas absolutas en ninguna otra celda.
***Se utiliza hasta el escenario S5 ya que hay datos sin comprender en el escenario S6***

In [1]:
# === 0. Configuración y carga de los escenarios ===

%matplotlib inline
import os


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# --- helpers del EDA: src2/ vive al lado del notebook ---
sys.path.insert(0, str(Path.cwd()))
from src2.data import (mostrar_dfs, eliminar_columnas, imputar_media,
                       partir, tipo_de, fase_de)
from src2.split import separar_variables, aplanar_ventanas
#from src2.train import *
import src2.train as train  
from src2.group import GroupedPCA
import src2.features as features

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
#from src2.reduce import BlockPca

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score,
                             roc_auc_score)



#rom google.colab import drive
#rive.mount("/content/gdrive/")
BASE =Path("/media/windows/Users/pablo/Documents/UNSA/4to/MetodosCuantitativos/TPFinal/tpfinal-colab/datasets/")


import pandas as pd

dfs = {
    "S0":     pd.read_csv(BASE / "datasetS0.csv"),
    "S1":     pd.read_csv(BASE / "datasetS1.csv"),
    "S1-alt": pd.read_csv(BASE / "datasetS1alt.csv"),
    "S2":     pd.read_csv(BASE / "datasetS2.csv"),
    "S3":     pd.read_csv(BASE / "datasetS3.csv"),
    "S4":     pd.read_csv(BASE / "datasetS4.csv"),
    "S5":     pd.read_csv(BASE / "datasetS5.csv"),
    "S6":     pd.read_csv(BASE / "datasetS6.csv"),
}
df = (pd.concat(dfs, names=["escenario"])
      .reset_index(level="escenario")
      .reset_index(drop=True))

/tmp/ipykernel_813710/530615980.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index(level="escenario")


In [2]:
print(df)

      escenario                         timestamp  attack_active attack_phase attack_phase_all  BSS-Inverter-AC-Meter.cos_phi1  BSS-Inverter-AC-Meter.cos_phi2  \
0            S0  2026-03-05 13:19:02.660000+00:00          False       normal           normal                       -0.310502                       -0.703539   
1            S0  2026-03-05 13:19:03.660000+00:00          False       normal           normal                       -0.310502                       -0.703539   
2            S0  2026-03-05 13:19:04.660000+00:00          False       normal           normal                       -0.310502                       -0.703539   
3            S0  2026-03-05 13:19:05.660000+00:00          False       normal           normal                       -0.310502                       -0.703539   
4            S0  2026-03-05 13:19:06.660000+00:00          False       normal           normal                       -0.999527                       -0.959250   
...         ...             

In [3]:
# --- agrupación de columnas: dispositivo, tipo, fase ---
SIG_TIPO = {
    "uln1": "Tensión fase-neutro", "uln2": "Tensión fase-neutro", "uln3": "Tensión fase-neutro",
    "ull12": "Tensión fase-fase", "ull23": "Tensión fase-fase", "ull31": "Tensión fase-fase",
    "iln1": "Corriente de fase", "iln2": "Corriente de fase", "iln3": "Corriente de fase",
    "i_sum3": "Corriente de neutro",
    "pln1": "Potencia activa", "pln2": "Potencia activa", "pln3": "Potencia activa", "p_sum3": "Potencia activa",
    "sln1": "Potencia aparente", "sln2": "Potencia aparente", "sln3": "Potencia aparente", "s_sum3": "Potencia aparente",
    "qln1": "Potencia reactiva", "qln2": "Potencia reactiva", "qln3": "Potencia reactiva", "q_sum3": "Potencia reactiva",
    "cos_phi1": "Factor de potencia", "cos_phi2": "Factor de potencia", "cos_phi3": "Factor de potencia",
    "freq": "Frecuencia",
    "phase_seq": "Secuencia de fases",
    "realtime": "Marca de tiempo del equipo",
}
UNIDAD = {
    "Tensión fase-neutro": "V", "Tensión fase-fase": "V", "Corriente de fase": "A",
    "Corriente de neutro": "A", "Potencia activa": "W", "Potencia aparente": "VA",
    "Potencia reactiva": "var", "Factor de potencia": "—", "Frecuencia": "Hz",
    "Secuencia de fases": "—", "Marca de tiempo del equipo": "s",
    "SOC": "fracción 0-1", "Señal de precio": "EUR/kWh", "Contexto de escenario": "—",
    "Control / etiquetas": "—", "Desconocido": "?",
}
FASE = {"ull12": "L1-L2", "ull23": "L2-L3", "ull31": "L3-L1"}

reg = []
for c in df.columns:
    dev, sig = partir(c)
    t = tipo_de(dev, sig)
    reg.append({"columna": c, "dispositivo": dev, "señal": sig,
                "tipo": t, "unidad": UNIDAD.get(t, "?"), "fase": fase_de(sig)})
cols = pd.DataFrame(reg)

print("columnas:", len(cols), "| dispositivos:", cols.dispositivo.nunique(), "| tipos:", cols.tipo.nunique())
print("\n--- por dispositivo ---")
por_dev = (cols.groupby("dispositivo")
           .agg(columnas=("columna", "size"), tipos=("tipo", lambda s: ", ".join(sorted(set(s)))))
           .sort_values("columnas", ascending=False))
display(por_dev)

columnas: 157 | dispositivos: 10 | tipos: 15

--- por dispositivo ---


,columnas,tipos
dispositivo,,
BSS-Inverter-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Load-B-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Substation-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Load-A-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
PV-Inverter-AC-Meter,28,"Corriente de fase, Corriente de neutro, Factor..."
Profile,8,Contexto de escenario
(sin prefijo),5,Control / etiquetas
MQTT-Price-Signal,2,Señal de precio
Battery-SOC-Reference,1,SOC


In [4]:
# # === 1. Integridad del registro y de las etiquetas ===

# # --- una fila por escenario: paso temporal, cobertura, monotonía y concordancia etiqueta/fase ---
# filas = []
# for k, datos in dfs.items():
#     t = pd.to_datetime(datos["timestamp"], errors="coerce")
#     paso = t.diff().dt.total_seconds().dropna()
#     es_normal = datos["attack_phase"].eq("normal")
#     activo = datos["attack_active"].astype(bool)
#     filas.append({
#         "escenario": k,
#         "filas": len(datos),
#         "inicio": t.min(),
#         "duracion_min": round((t.max() - t.min()).total_seconds() / 60, 1),
#         "paso_s": paso.mode().iloc[0],
#         "pasos_distintos": paso.nunique(),
#         "timestamp_nulo": int(t.isna().sum()),
#         "monotonico": t.is_monotonic_increasing,
#         "positivos": int(activo.sum()),
#         "concuerda_con_fase": bool((activo == ~es_normal).all()),
#     })

# auditoria = pd.DataFrame(filas).set_index("escenario")
# print("auditoría del registro crudo, una fila por escenario")
# display(auditoria)


## 2. Datos Nulos

En este release **una celda vacía no es un cero ni un `NaN` imputado: es ausencia de valor de origen**.

In [5]:
# --- vacíos por columna y por fila ---

num = df.select_dtypes(include=[np.number])
no_num = [c for c in df.columns if c not in num.columns]
miss = df.isna()

print(f"columnas numéricas: {len(num.columns)} | no numéricas: {len(no_num)}")
if no_num:
    print("no numéricas:", no_num)

print(f"celdas vacías: {int(miss.values.sum()):,} de {miss.size:,} ({miss.values.mean()*100:.2f} %)")
print(f"filas con algún vacío: {int((miss.sum(axis=1) > 0).sum()):,} de {len(miss):,}")

por_col = miss.mean().mul(100).sort_values(ascending=False)
con_vacios = por_col[por_col > 0]
print(f"ncolumnas con al menos un vacío: {len(con_vacios)} de {len(por_col)}")
print("\ntop 15 columnas más incompletas (%):")
print(con_vacios.head(15).round(2).to_string())
#print(con_vacios.index.tolist())



columnas numéricas: 150 | no numéricas: 7
no numéricas: ['escenario', 'timestamp', 'attack_active', 'attack_phase', 'attack_phase_all', 'Profile.timestamp', 'MQTT-Price-Signal.event']
celdas vacías: 20,123 de 2,253,578 (0.89 %)
filas con algún vacío: 12,992 de 14,354
ncolumnas con al menos un vacío: 152 de 157

top 15 columnas más incompletas (%):
MQTT-Price-Signal.event                90.51
MQTT-Price-Signal.price_eur_per_kwh     6.77
Battery-SOC-Reported.soc_value          1.40
Profile.total_irradiance                1.37
Profile.pv_active_kw                    1.37
Profile.load1_kw                        1.37
Profile.pv_input_kw                     1.37
Profile.load2_kw                        1.37
Profile.ambient_temperature             1.37
Profile.pv_reactive_kvar                1.37
Profile.timestamp                       1.37
Battery-SOC-Reference.soc_value         0.35
BSS-Inverter-AC-Meter.i_sum3            0.22
BSS-Inverter-AC-Meter.qln3              0.22
BSS-Inverter-AC-Mete

 # Eliminar columnas sin aporte
 por ahora vamos a descartar las columas que involucren marcas temporales y etapas del ataque ya que no estamos interezados en analizar estas caracteristicas.
"

In [6]:
df=eliminar_columnas(df)

# Imputacion de datos 

In [7]:
# --- columnas con nulos por escenario, antes de imputar ---
df.loc[:, df.isna().any()]
# print("columnas con nulos por escenario:", {k: df.shape[1] for k, df in dfs_con_nulos.items()})
# mostrar_dfs({"S4": dfs_con_nulos["S4"]}, 3)

,BSS-Inverter-AC-Meter.cos_phi1,BSS-Inverter-AC-Meter.cos_phi2,BSS-Inverter-AC-Meter.cos_phi3,BSS-Inverter-AC-Meter.freq,BSS-Inverter-AC-Meter.i_sum3,BSS-Inverter-AC-Meter.iln1,BSS-Inverter-AC-Meter.iln2,BSS-Inverter-AC-Meter.iln3,BSS-Inverter-AC-Meter.p_sum3,BSS-Inverter-AC-Meter.pln1,BSS-Inverter-AC-Meter.pln2,BSS-Inverter-AC-Meter.pln3,BSS-Inverter-AC-Meter.q_sum3,BSS-Inverter-AC-Meter.qln1,BSS-Inverter-AC-Meter.qln2,BSS-Inverter-AC-Meter.qln3,BSS-Inverter-AC-Meter.s_sum3,BSS-Inverter-AC-Meter.sln1,BSS-Inverter-AC-Meter.sln2,BSS-Inverter-AC-Meter.sln3,BSS-Inverter-AC-Meter.ull12,BSS-Inverter-AC-Meter.ull23,BSS-Inverter-AC-Meter.ull31,BSS-Inverter-AC-Meter.uln1,BSS-Inverter-AC-Meter.uln2,BSS-Inverter-AC-Meter.uln3,Battery-SOC-Reference.soc_value,Battery-SOC-Reported.soc_value,Load-A-AC-Meter.cos_phi1,Load-A-AC-Meter.cos_phi2,...,PV-Inverter-AC-Meter.ull31,PV-Inverter-AC-Meter.uln1,PV-Inverter-AC-Meter.uln2,PV-Inverter-AC-Meter.uln3,Substation-AC-Meter.cos_phi1,Substation-AC-Meter.cos_phi2,Substation-AC-Meter.cos_phi3,Substation-AC-Meter.freq,Substation-AC-Meter.i_sum3,Substation-AC-Meter.iln1,Substation-AC-Meter.iln2,Substation-AC-Meter.iln3,Substation-AC-Meter.p_sum3,Substation-AC-Meter.pln1,Substation-AC-Meter.pln2,Substation-AC-Meter.pln3,Substation-AC-Meter.q_sum3,Substation-AC-Meter.qln1,Substation-AC-Meter.qln2,Substation-AC-Meter.qln3,Substation-AC-Meter.s_sum3,Substation-AC-Meter.sln1,Substation-AC-Meter.sln2,Substation-AC-Meter.sln3,Substation-AC-Meter.ull12,Substation-AC-Meter.ull23,Substation-AC-Meter.ull31,Substation-AC-Meter.uln1,Substation-AC-Meter.uln2,Substation-AC-Meter.uln3
0,-0.310502,-0.703539,-0.989501,50.008049,0.101279,0.394920,0.377298,0.421309,-53.832207,-18.320284,-16.559868,-18.952057,28.835691,-2.792666,-9.061968,3.146089,281.451599,93.153397,88.853355,99.444839,407.938171,408.500793,408.841309,235.879349,235.498993,236.037918,NaN,NaN,-0.778859,0.997305,...,410.934448,237.135132,236.710876,237.264816,-0.965705,-0.966816,-0.962146,50.005825,1.437780,9.847172,9.582252,9.602752,-5902.140137,-1999.145142,-1944.466309,-1958.528809,1616.824585,541.077271,517.283081,558.464233,6802.083984,2306.897217,2242.235840,2252.951416,405.437134,405.948944,406.069855,234.270035,233.998840,234.615189
1,-0.310502,-0.703539,-0.989501,50.008049,0.101279,0.394920,0.377298,0.421309,-53.832207,-18.320284,-16.559868,-18.952057,28.835691,-2.792666,-9.061968,3.146089,281.451599,93.153397,88.853355,99.444839,407.938171,408.500793,408.841309,235.879349,235.498993,236.037918,0.199597,NaN,-0.778859,0.997305,...,410.934448,237.135132,236.710876,237.264816,-0.965705,-0.966816,-0.962146,50.005825,1.437780,9.847172,9.582252,9.602752,-5902.140137,-1999.145142,-1944.466309,-1958.528809,1616.824585,541.077271,517.283081,558.464233,6802.083984,2306.897217,2242.235840,2252.951416,405.437134,405.948944,406.069855,234.270035,233.998840,234.615189
2,-0.310502,-0.703539,-0.989501,50.008049,0.101279,0.394920,0.377298,0.421309,-53.832207,-18.320284,-16.559868,-18.952057,28.835691,-2.792666,-9.061968,3.146089,281.451599,93.153397,88.853355,99.444839,407.938171,408.500793,408.841309,235.879349,235.498993,236.037918,0.199597,NaN,-0.778859,0.997305,...,410.934448,237.135132,236.710876,237.264816,-0.965705,-0.966816,-0.962146,50.005825,1.437780,9.847172,9.582252,9.602752,-5902.140137,-1999.145142,-1944.466309,-1958.528809,1616.824585,541.077271,517.283081,558.464233,6802.083984,2306.897217,2242.235840,2252.951416,405.437134,405.948944,406.069855,234.270035,233.998840,234.615189
3,-0.310502,-0.703539,-0.989501,50.008049,0.101279,0.394920,0.377298,0.421309,-53.832207,-18.320284,-16.559868,-18.952057,28.835691,-2.792666,-9.061968,3.146089,281.451599,93.153397,88.853355,99.444839,407.938171,408.500793,408.841309,235.879349,235.498993,236.037918,0.199597,NaN,-0.778859,0.997305,...,410.934448,237.135132,236.710876,237.264816,-0.965705,-0.966816,-0.962146,50.005825,1.437780,9.847172,9.582252,9.602752,-5902.140137,-1999.145142,-1944.466309,-1958.528809,161

In [8]:
import src2.graphics as graphics

In [9]:
# --- imputar la media por escenario y mostrar el resultado, no el estado previo ---
df =imputar_media(df)
print("nulos restantes:", {k: int(df.isna().sum().sum()) for k, df in dfs.items()})

nulos restantes: {'S0': 482, 'S1': 563, 'S1-alt': 286, 'S2': 284, 'S3': 269, 'S4': 4417, 'S5': 211, 'S6': 1057}


# Split de Datos
Debido a la naturaleza temporal e interdependiente de las variables de telemetría ciberfísica, aplicar una división aleatoria convencional (*random train-test split*) introduce el riesgo de data leakage y overfitting, ya que el modelo tiende a memorizar tendencias continuas o correlaciones contiguas dentro de un mismo episodio operacional. Para capturar la dinámica secuencial de los sensores sin contaminar el entrenamiento, los datos se estructuraron mediante rolling windows de 20 pasos temporales hacia atrás. Asimismo, la validación del modelo se diseñó bajo un esquema de Leave-One-Scenario-Out(LOSO), lo que garantiza que el conjunto de prueba contenga únicamente eventos de ataque u operación completamente inéditos, evaluando con rigor la capacidad real de generalización del sistema ante situaciones no vistas

In [10]:
dfX = df.drop(["attack_active"],axis=1)
dfY=  df["attack_active"]

In [11]:
print(dfX.head(5))
print(dfY.head(5))

  escenario  BSS-Inverter-AC-Meter.cos_phi1  BSS-Inverter-AC-Meter.cos_phi2  BSS-Inverter-AC-Meter.cos_phi3  BSS-Inverter-AC-Meter.freq  BSS-Inverter-AC-Meter.i_sum3  \
0        S0                       -0.310502                       -0.703539                       -0.989501                   50.008049                      0.101279   
1        S0                       -0.310502                       -0.703539                       -0.989501                   50.008049                      0.101279   
2        S0                       -0.310502                       -0.703539                       -0.989501                   50.008049                      0.101279   
3        S0                       -0.310502                       -0.703539                       -0.989501                   50.008049                      0.101279   
4        S0                       -0.999527                       -0.959250                       -0.969011                   50.004364                    

In [12]:
# # === 3. Aplanado de ventanas: 20 pasos x N canales por ventana ===
# VENTANA = 20

# # --- se aplana escenario por escenario: ninguna ventana cruza el final de uno con el inicio del siguiente ---
# dfsX_flat, dfsY_flat = {}, {}
# for k in dfsX:
#     dfsX_flat[k], dfsY_flat[k] = aplanar_ventanas(dfsX[k], dfsY[k],VENTANA)

# primero = next(iter(dfsX_flat))
# print("columnas por ventana:", dfsX_flat[primero].shape[1])
# dfsX=dfsX_flat
# dfsY=dfsY_flat

In [14]:
# dfrm=features.rango_movil(dfX)

dfsX = {k: g.drop(columns="escenario") for k, g in dfX.groupby("escenario", sort=False)}

rango = features.rango_escenarios(dfsX, ventana=120)
dfrm = (pd.concat(rango, names=["escenario"])
          .reset_index(level="escenario")
          .reset_index(drop=True))

In [16]:
train.rf_logo(dfrm.drop(columns="escenario"),dfY,dfrm["escenario"])

,escenario,features,n,positivos,accuracy,precision,recall,f1,especificidad,auc,recall_5
fold,,,,,,,,,,,
1,S0,132,1798.00,0.00,0.515,0.000,0.000,0.000,0.515,NaN,NaN
2,S1,132,1798.00,913.00,0.796,0.790,0.814,0.802,0.777,0.858,0.251
3,S1-alt,132,1792.00,910.00,0.777,0.727,0.899,0.804,0.652,0.838,0.358
4,S2,132,1768.00,903.00,0.605,0.964,0.235,0.378,0.991,0.892,0.375
5,S3,132,1800.00,912.00,0.818,0.953,0.673,0.789,0.966,0.880,0.765
6,S4,132,1798.00,1257.00,0.913,0.997,0.878,0.934,0.994,0.975,0.893
7,S5,132,1800.00,1341.00,0.801,0.958,0.766,0.851,0.902,0.929,0.302
8,S6,132,1800.00,1350.00,0.608,0.904,0.534,0.671,0.829,0.767,0.243
mean,NaN,132,1794.25,948.25,0.729,0.787,0.600,0.654,0.828,0.877,0.455


In [17]:
train.rf_block_logo(dfrm.drop(columns="escenario"),dfY,dfrm["escenario"])

,escenario,features,n,positivos,accuracy,precision,recall,f1,especificidad,auc,recall_5
fold,,,,,,,,,,,
1,S0,36,1798.00,0.00,0.526,0.000,0.000,0.000,0.526,NaN,NaN
2,S1,37,1798.00,913.00,0.806,0.794,0.834,0.813,0.777,0.855,0.206
3,S1-alt,37,1792.00,910.00,0.781,0.730,0.902,0.807,0.655,0.868,0.476
4,S2,38,1768.00,903.00,0.482,0.341,0.016,0.030,0.969,0.918,0.483
5,S3,36,1800.00,912.00,0.766,0.868,0.634,0.733,0.901,0.884,0.582
6,S4,33,1798.00,1257.00,0.933,0.997,0.906,0.950,0.994,0.986,0.951
7,S5,34,1800.00,1341.00,0.815,0.959,0.785,0.863,0.902,0.926,0.309
8,S6,37,1800.00,1350.00,0.582,1.000,0.442,0.613,1.000,0.941,0.781
mean,NaN,36,1794.25,948.25,0.711,0.711,0.565,0.601,0.840,0.911,0.541
